# Deduplication: Exact, Latest-Wins, Fuzzy, Survivorship

## What you'll practice

Duplicates come from retries, re-deliveries, CDC replays, and humans typing the same customer twice. You'll handle exact dupes, versioned records (latest wins), reporting,
fuzzy matching with **blocking**, and golden-record survivorship rules.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_04_deduplication.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/04_deduplication.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Exact duplicate rows

**Implement:** `dedupe_exact(rows: list[dict]) -> tuple[list[dict], int]`

Remove rows that are identical (same keys and values, key order irrelevant), keeping the first. Return `(unique_rows, removed_count)`. Values are hashable.

**Input:** `rows: list[dict]`

**Expected output:** `tuple[list[dict], int]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_dedupe_exact` in `tests/etl/test_08_04_deduplication.py`.

**Check:** `pytest tests/etl/test_08_04_deduplication.py -k test_dedupe_exact`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Latest version per business key

**Implement:** `dedupe_latest(rows: list[dict], key: str | tuple[str, ...], order_by: str) -> list[dict]`

Keep one row per key (a field name or a tuple of field names), choosing the greatest `order_by` value; ties → the later row in input. Preserve the order in which keys first appeared.

**Input:** `rows: list[dict], key: str | tuple[str, ...], order_by: str`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_dedupe_latest, test_dedupe_latest_composite` in `tests/etl/test_08_04_deduplication.py`.

**Check:** `pytest tests/etl/test_08_04_deduplication.py -k "test_dedupe_latest or test_dedupe_latest_composite"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Duplicate report

**Implement:** `duplicate_report(rows: list[dict], key: str) -> dict`

Return `{"rows", "distinct_keys", "duplicate_keys", "duplicate_rows" (rows beyond the first per key), "worst": [(key, count) top 5 by count desc then key]}`. Run it on `datasets/csv/orders.csv` by `order_id`.

**Input:** `rows: list[dict], key: str`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_duplicate_report, test_duplicate_report_real` in `tests/etl/test_08_04_deduplication.py`.

**Check:** `pytest tests/etl/test_08_04_deduplication.py -k "test_duplicate_report or test_duplicate_report_real"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Fuzzy duplicates with blocking

**Implement:** `fuzzy_duplicates(customers: list[dict], threshold: float = 0.9) -> list[tuple[str, str, float]]`

Find probable duplicate customers whose **normalized names** are similar (`difflib.SequenceMatcher(None, a, b).ratio() >= threshold`). Normalize: lowercase, strip accents
(`unicodedata.normalize("NFKD", ...)` and drop combining marks), remove non-letters, collapse spaces. To avoid O(n²) across everyone, only compare customers in the same **block** =
`(country, first letter of normalized name)`. Return `(id_a, id_b, ratio rounded 3)` with `id_a < id_b`, sorted by ratio desc then ids.

**Input:** `customers: list[dict], threshold: float = 0.9`

**Expected output:** `list[tuple[str, str, float]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
normalize_name('  Jörg  MÜLLER-Smith ')
# -> 'jorg muller smith'
```

**Check:** `pytest tests/etl/test_08_04_deduplication.py -k "test_normalize_name or test_fuzzy_duplicates"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Golden records with survivorship rules · 🧩 Mini-project

**Implement:** `build_golden_records(customers: list[dict]) -> list[dict]`

Merge duplicate customers (same normalized email: strip + lower; rows without email are their own group keyed by `customer_id`) into one golden record per group:
- `golden_id`: the smallest `customer_id`; `source_ids`: all ids sorted
- `name`: the longest normalized (whitespace-collapsed, title-cased) non-blank name; ties → from the most recent signup
- `email`: normalized; `country`: most common upper-cased value (ties → alphabetical)
- `signup_date`: the earliest valid date (`YYYY-MM-DD` or `DD/MM/YYYY`), as ISO; `is_active`: `True` if any source is active
- `segment`: priority enterprise > smb > retail (highest present)
Return sorted by `golden_id`. Run it on `datasets/csv/customers.csv` and compare counts before/after.

**Input:** `customers: list[dict]`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_build_golden_records, test_build_golden_records_real` in `tests/etl/test_08_04_deduplication.py`.

**Check:** `pytest tests/etl/test_08_04_deduplication.py -k "test_build_golden_records or test_build_golden_records_real"`

**YOUR CODE ↓**